# Orbit Wars - 試行錯誤の記録

## 1. 環境のセットアップ

In [1]:
%%capture
!pip install --upgrade "kaggle-environments>=1.28.0"

from kaggle_environments import make
import math

env = make("orbit_wars", debug=True)
print(f"Environment: {env.name} v{env.version}")
print(f"Players: {env.specification.agents}")
print(f"Max steps: {env.configuration.episodeSteps}")

## 2. ゲーム理解

In [2]:
# Run a quick game to see what the observation looks like
env = make("orbit_wars", debug=True)
env.run(["random", "random"])

# Peek at the initial observation
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet

obs = env.steps[1][0].observation  # step 1 = first action step
planets = [Planet(*p) for p in obs.planets]
print(f"Player: {obs.player}")
print(f"Angular velocity: {obs.angular_velocity:.4f} rad/turn")
print(f"\nPlanets ({len(planets)}):")
for p in planets[:6]:
    owner_str = f"Player {p.owner}" if p.owner >= 0 else "Neutral"
    print(f"  id={p.id} owner={owner_str:10s} pos=({p.x:.1f}, {p.y:.1f}) r={p.radius:.1f} ships={p.ships} prod={p.production}")

Player: 0
Angular velocity: 0.0340 rad/turn

Planets (20):
  id=0 owner=Neutral    pos=(83.0, 86.7) r=2.6 ships=55 prod=5
  id=1 owner=Neutral    pos=(13.3, 83.0) r=2.6 ships=55 prod=5
  id=2 owner=Neutral    pos=(86.7, 17.0) r=2.6 ships=55 prod=5
  id=3 owner=Neutral    pos=(17.0, 13.3) r=2.6 ships=55 prod=5
  id=4 owner=Neutral    pos=(79.3, 98.3) r=1.0 ships=64 prod=1
  id=5 owner=Neutral    pos=(1.7, 79.3) r=1.0 ships=64 prod=1


## 3. エージェントの実装

In [3]:
# ============================================================
# agent_v1：一番近い惑星を狙うだけの基本エージェント
# スコア：約400
# 問題点：距離だけで判断するため生産力の低い惑星を狙いがち
# ============================================================
def agent_v1(obs, config=None):
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]
    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]
    if not targets:
        return moves
    for mine in my_planets:
        nearest = min(targets, key=lambda t: math.hypot(mine.x-t.x, mine.y-t.y))
        ships_needed = max(nearest.ships + 1, 20)
        if mine.ships >= ships_needed:
            angle = math.atan2(nearest.y-mine.y, nearest.x-mine.x)
            moves.append([mine.id, angle, ships_needed])
    return moves

In [4]:
# ============================================================
# agent_v2：距離÷生産力で優先度をつけたエージェント
# スコア：465
# 改善点：生産力が高くて近い惑星を優先するようにした
# 改善点：守備用に最低10隻（GUARD）を残すようにした
# 問題点：最低20隻送るため攻撃が遅い
# ============================================================
def agent_v2(obs, config=None):
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]
    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]
    if not targets:
        return moves
    for mine in my_planets:
        def priority(t):
            dist = math.hypot(mine.x-t.x, mine.y-t.y)
            return dist / (t.production + 1)
        nearest = min(targets, key=priority)
        GUARD = 10
        ships_needed = max(nearest.ships + 1, 20)
        if mine.ships >= ships_needed + GUARD:
            angle = math.atan2(nearest.y-mine.y, nearest.x-mine.x)
            moves.append([mine.id, angle, ships_needed])
    return moves

In [5]:
# ============================================================
# agent_v3：静止惑星のみを狙うエージェント
# スコア：600前後
# 改善点：自転惑星を狙うと艦隊がすり抜けることを発見
#         → 静止惑星のみに絞ることで艦隊の無駄をなくした
# 問題点：自転惑星（内側の生産力高い惑星）を全く取れない
#         → 実際の対戦では内側を先に取られて負けることが多い
# ============================================================
def agent_v3(obs, config=None):
    import math
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]
    angular_velocity = obs.get("angular_velocity", 0)

    my_planets = [p for p in planets if p.owner == player]
    
    cx, cy = 50, 50
    # 静止惑星だけを狙う
    targets = [p for p in planets 
               if p.owner != player
               and math.hypot(p.x-cx, p.y-cy) + p.radius >= 50]

    if not targets:
        # 静止惑星がなければ全部対象
        targets = [p for p in planets if p.owner != player]

    for mine in my_planets:
        def priority(t):
            dist = math.hypot(mine.x-t.x, mine.y-t.y)
            return dist / (t.production + 1)
        nearest = min(targets, key=priority)
        GUARD = 10
        ships_needed = max(nearest.ships + 1, 8)
        if mine.ships >= ships_needed + GUARD:
            angle = math.atan2(nearest.y-mine.y, nearest.x-mine.x)
            moves.append([mine.id, angle, ships_needed])

    return moves

In [6]:
# ============================================================
# agent_v4：少数艦隊で素早く広げる
# スコア：650前後（現在の最強版）
# 改善点：ships=8に下げて素早く多くの惑星を確保できるようにした
# ============================================================
def agent_v4(obs, config=None):
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]

    my_planets = [p for p in planets if p.owner == player]
    
    cx, cy = 50, 50
    # 静止惑星だけを狙う
    targets = [p for p in planets 
               if p.owner != player
               and math.hypot(p.x-cx, p.y-cy) + p.radius >= 50]

    if not targets:
        # 静止惑星がなければ全部対象
        targets = [p for p in planets if p.owner != player]

    for mine in my_planets:
        def priority(t):
            dist = math.hypot(mine.x-t.x, mine.y-t.y)
            return dist / (t.production + 1)
        nearest = min(targets, key=priority)
        GUARD = 10
        ships_needed = max(nearest.ships + 1, 8)
        if mine.ships >= ships_needed + GUARD:
            angle = math.atan2(nearest.y-mine.y, nearest.x-mine.x)
            moves.append([mine.id, angle, ships_needed])

    return moves

In [7]:
def agent_v5a(obs, config=None):
    import math
    moves = []

    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]
    angular_velocity = obs.get("angular_velocity", 0)

    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]

    if not targets:
        return moves

    cx, cy = 50, 50

    for mine in my_planets:

        # ここだけ変更：上位コードのスコア計算を取り込む
        def priority(t):
            dist = math.hypot(mine.x-t.x, mine.y-t.y)
            ships_needed = max(t.ships + 1, 8)

            # 艦隊速度を正確に計算
            fleet_speed = 1.0 + 5.0 * (math.log(max(1, ships_needed)) / math.log(1000)) ** 1.5
            
            # 到着までの時間
            eta = dist / fleet_speed

            # 敵惑星なら到着までに増産する分を考慮
            enemy_produced = eta * t.production if t.owner != -1 else 0
            enemy_bonus = t.production if t.owner != -1 else 0

            score = (
                (100 - dist)                          # 近いほど良い
                + (30 * t.production)                 # 生産力高いほど良い
                + (20 * enemy_bonus)                  # 敵惑星は価値が高い
                - (0.7 * (t.ships + enemy_produced))  # 必要な船が多いほど悪い
                - (2 * eta)                           # 到着が遅いほど悪い
            )
            return -score  # 大きいほど優先なので符号反転

        nearest = min(targets, key=priority)
        ships_needed = max(nearest.ships + 1, 8)

        # 自転惑星かどうか判定
        dist_from_sun = math.hypot(nearest.x-cx, nearest.y-cy)
        is_orbiting = dist_from_sun + nearest.radius < 50

        if is_orbiting:
            turns = int(math.hypot(mine.x-nearest.x, mine.y-nearest.y) / 3) + 1
            dx = nearest.x - cx
            dy = nearest.y - cy
            r = math.sqrt(dx**2 + dy**2)
            cur_angle = math.atan2(dy, dx)
            fut_angle = cur_angle + angular_velocity * turns
            target_x = cx + r * math.cos(fut_angle)
            target_y = cy + r * math.sin(fut_angle)
        else:
            target_x, target_y = nearest.x, nearest.y

        GUARD = 10
        if mine.ships >= ships_needed + GUARD:
            angle = math.atan2(target_y - mine.y, target_x - mine.x)
            moves.append([mine.id, angle, ships_needed])

    return moves

# パターンB：距離を重視
def agent_v5b(obs, config=None):
    import math
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]
    angular_velocity = obs.get("angular_velocity", 0)
    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]
    if not targets:
        return moves
    cx, cy = 50, 50
    for mine in my_planets:
        def priority(t):
            dist = math.hypot(mine.x-t.x, mine.y-t.y)
            ships_needed = max(t.ships + 1, 8)
            fleet_speed = 1.0 + 5.0 * (math.log(max(1, ships_needed)) / math.log(1000)) ** 1.5
            eta = dist / fleet_speed
            enemy_produced = eta * t.production if t.owner != -1 else 0
            enemy_bonus = t.production if t.owner != -1 else 0
            score = (
                (200 - dist * 2)       # 距離の影響を2倍
                + (15 * t.production)
                + (10 * enemy_bonus)
                - (0.7 * (t.ships + enemy_produced))
                - (2 * eta)
            )
            return -score
        nearest = min(targets, key=priority)
        ships_needed = max(nearest.ships + 1, 8)
        dist_from_sun = math.hypot(nearest.x-cx, nearest.y-cy)
        is_orbiting = dist_from_sun + nearest.radius < 50
        if is_orbiting:
            turns = int(math.hypot(mine.x-nearest.x, mine.y-nearest.y) / 3) + 1
            dx = nearest.x - cx
            dy = nearest.y - cy
            r = math.sqrt(dx**2 + dy**2)
            fut_angle = math.atan2(dy, dx) + angular_velocity * turns
            target_x = cx + r * math.cos(fut_angle)
            target_y = cy + r * math.sin(fut_angle)
        else:
            target_x, target_y = nearest.x, nearest.y
        GUARD = 10
        if mine.ships >= ships_needed + GUARD:
            angle = math.atan2(target_y - mine.y, target_x - mine.x)
            moves.append([mine.id, angle, ships_needed])
    return moves

# パターンC：到着時間を重視
def agent_v5c(obs, config=None):
    import math
    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]
    angular_velocity = obs.get("angular_velocity", 0)
    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]
    if not targets:
        return moves
    cx, cy = 50, 50
    for mine in my_planets:
        def priority(t):
            dist = math.hypot(mine.x-t.x, mine.y-t.y)
            ships_needed = max(t.ships + 1, 8)
            fleet_speed = 1.0 + 5.0 * (math.log(max(1, ships_needed)) / math.log(1000)) ** 1.5
            eta = dist / fleet_speed
            enemy_produced = eta * t.production if t.owner != -1 else 0
            enemy_bonus = t.production if t.owner != -1 else 0
            score = (
                (100 - dist)
                + (15 * t.production)
                + (10 * enemy_bonus)
                - (0.7 * (t.ships + enemy_produced))
                - (5 * eta)            # 2→5に増加
            )
            return -score
        nearest = min(targets, key=priority)
        ships_needed = max(nearest.ships + 1, 8)
        dist_from_sun = math.hypot(nearest.x-cx, nearest.y-cy)
        is_orbiting = dist_from_sun + nearest.radius < 50
        if is_orbiting:
            turns = int(math.hypot(mine.x-nearest.x, mine.y-nearest.y) / 3) + 1
            dx = nearest.x - cx
            dy = nearest.y - cy
            r = math.sqrt(dx**2 + dy**2)
            fut_angle = math.atan2(dy, dx) + angular_velocity * turns
            target_x = cx + r * math.cos(fut_angle)
            target_y = cy + r * math.sin(fut_angle)
        else:
            target_x, target_y = nearest.x, nearest.y
        GUARD = 10
        if mine.ships >= ships_needed + GUARD:
            angle = math.atan2(target_y - mine.y, target_x - mine.x)
            moves.append([mine.id, angle, ships_needed])
    return moves

def make_v5_variant(prod_mult, enemy_mult, ships_mult):
    def agent_test(obs, config=None):
        import math
        moves = []
        player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
        raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
        planets = [Planet(*p) for p in raw_planets]
        angular_velocity = obs.get("angular_velocity", 0)
        my_planets = [p for p in planets if p.owner == player]
        targets = [p for p in planets if p.owner != player]
        if not targets:
            return moves
        cx, cy = 50, 50
        for mine in my_planets:
            def priority(t):
                dist = math.hypot(mine.x-t.x, mine.y-t.y)
                ships_needed = max(t.ships + 1, 8)
                fleet_speed = 1.0 + 5.0 * (math.log(max(1, ships_needed)) / math.log(1000)) ** 1.5
                eta = dist / fleet_speed
                enemy_produced = eta * t.production if t.owner != -1 else 0
                enemy_bonus = t.production if t.owner != -1 else 0
                score = (
                    (100 - dist)
                    + (prod_mult * t.production)
                    + (enemy_mult * enemy_bonus)
                    - (ships_mult * (t.ships + enemy_produced))
                    - (2 * eta)
                )
                return -score
            nearest = min(targets, key=priority)
            ships_needed = max(nearest.ships + 1, 8)
            dist_from_sun = math.hypot(nearest.x-cx, nearest.y-cy)
            is_orbiting = dist_from_sun + nearest.radius < 50
            if is_orbiting:
                turns = int(math.hypot(mine.x-nearest.x, mine.y-nearest.y) / 3) + 1
                dx = nearest.x - cx
                dy = nearest.y - cy
                r = math.sqrt(dx**2 + dy**2)
                fut_angle = math.atan2(dy, dx) + angular_velocity * turns
                target_x = cx + r * math.cos(fut_angle)
                target_y = cy + r * math.sin(fut_angle)
            else:
                target_x, target_y = nearest.x, nearest.y
            GUARD = 10
            if mine.ships >= ships_needed + GUARD:
                angle = math.atan2(target_y - mine.y, target_x - mine.x)
                moves.append([mine.id, angle, ships_needed])
        return moves
    return agent_test

# 3パターン作成
agent_v5a2 = make_v5_variant(prod_mult=50, enemy_mult=20, ships_mult=0.7)
agent_v5a3 = make_v5_variant(prod_mult=30, enemy_mult=40, ships_mult=0.7)
agent_v5a4 = make_v5_variant(prod_mult=30, enemy_mult=20, ships_mult=0.3)

In [8]:
# ============================================================
# agent_v6：到着までの増産を考慮した必要船数計算
# 改善点：敵が到着までに増産する分を ships_needed に加算
# ============================================================
def agent_v6(obs, config=None):
    import math
    moves = []

    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]
    angular_velocity = obs.get("angular_velocity", 0)

    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player]

    if not targets:
        return moves

    cx, cy = 50, 50

    for mine in my_planets:
        def priority(t):
            dist = math.hypot(mine.x-t.x, mine.y-t.y)
            ships_needed = max(t.ships + 1, 8)
            fleet_speed = 1.0 + 5.0 * (math.log(max(1, ships_needed)) / math.log(1000)) ** 1.5
            eta = dist / fleet_speed
            enemy_produced = eta * t.production if t.owner != -1 else 0
            enemy_bonus = t.production if t.owner != -1 else 0
            score = (
                (100 - dist)
                + (30 * t.production)
                + (20 * enemy_bonus)
                - (0.7 * (t.ships + enemy_produced))
                - (2 * eta)
            )
            return -score

        nearest = min(targets, key=priority)

        # ここが変更点：到着までの増産を考慮
        base_ships = nearest.ships + 1
        fleet_speed = 1.0 + 5.0 * (math.log(max(1, base_ships)) / math.log(1000)) ** 1.5
        dist = math.hypot(mine.x-nearest.x, mine.y-nearest.y)
        eta = dist / fleet_speed

        # 敵惑星なら到着までに増産する分を加算
        extra = int(eta * nearest.production) if nearest.owner != -1 else 0
        ships_needed = max(base_ships + extra, 8)

        dist_from_sun = math.hypot(nearest.x-cx, nearest.y-cy)
        is_orbiting = dist_from_sun + nearest.radius < 50

        if is_orbiting:
            turns = int(dist / 3) + 1
            dx = nearest.x - cx
            dy = nearest.y - cy
            r = math.sqrt(dx**2 + dy**2)
            cur_angle = math.atan2(dy, dx)
            fut_angle = cur_angle + angular_velocity * turns
            target_x = cx + r * math.cos(fut_angle)
            target_y = cy + r * math.sin(fut_angle)
        else:
            target_x, target_y = nearest.x, nearest.y

        GUARD = 10
        if mine.ships >= ships_needed + GUARD:
            angle = math.atan2(target_y - mine.y, target_x - mine.x)
            moves.append([mine.id, angle, ships_needed])

    return moves

In [9]:
# ============================================================
# agent_v7：彗星を攻撃対象から除外
# 改善点：comet_planet_idsで彗星を除外
#         彗星は動くので狙っても無駄になりやすい
# ============================================================
def agent_v7(obs, config=None):
    import math
    moves = []

    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]
    angular_velocity = obs.get("angular_velocity", 0)
    comet_ids = set(obs.get("comet_planet_ids", []) if isinstance(obs, dict) else obs.comet_planet_ids)

    my_planets = [p for p in planets if p.owner == player]

    # 彗星を除外した攻撃対象
    targets = [p for p in planets
               if p.owner != player
               and p.id not in comet_ids]

    if not targets:
        # 彗星しかなければ彗星も対象にする
        targets = [p for p in planets if p.owner != player]

    cx, cy = 50, 50

    for mine in my_planets:
        def priority(t):
            dist = math.hypot(mine.x-t.x, mine.y-t.y)
            ships_needed = max(t.ships + 1, 8)
            fleet_speed = 1.0 + 5.0 * (math.log(max(1, ships_needed)) / math.log(1000)) ** 1.5
            eta = dist / fleet_speed
            enemy_produced = eta * t.production if t.owner != -1 else 0
            enemy_bonus = t.production if t.owner != -1 else 0
            score = (
                (100 - dist)
                + (30 * t.production)
                + (20 * enemy_bonus)
                - (0.7 * (t.ships + enemy_produced))
                - (2 * eta)
            )
            return -score

        nearest = min(targets, key=priority)

        base_ships = nearest.ships + 1
        fleet_speed = 1.0 + 5.0 * (math.log(max(1, base_ships)) / math.log(1000)) ** 1.5
        dist = math.hypot(mine.x-nearest.x, mine.y-nearest.y)
        eta = dist / fleet_speed
        extra = int(eta * nearest.production) if nearest.owner != -1 else 0
        ships_needed = max(base_ships + extra, 8)

        dist_from_sun = math.hypot(nearest.x-cx, nearest.y-cy)
        is_orbiting = dist_from_sun + nearest.radius < 50

        if is_orbiting:
            turns = int(dist / 3) + 1
            dx = nearest.x - cx
            dy = nearest.y - cy
            r = math.sqrt(dx**2 + dy**2)
            cur_angle = math.atan2(dy, dx)
            fut_angle = cur_angle + angular_velocity * turns
            target_x = cx + r * math.cos(fut_angle)
            target_y = cy + r * math.sin(fut_angle)
        else:
            target_x, target_y = nearest.x, nearest.y

        GUARD = 10
        if mine.ships >= ships_needed + GUARD:
            angle = math.atan2(target_y - mine.y, target_x - mine.x)
            moves.append([mine.id, angle, ships_needed])

    return moves

In [10]:
# ============================================================
# agent_v8：既に艦隊が向かっている惑星への二重攻撃を防ぐ
# 改善点：fleetを見て既に攻撃中の惑星は優先度を下げる
# ============================================================
def agent_v8(obs, config=None):
    import math
    moves = []

    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    raw_fleets = obs.get("fleets", []) if isinstance(obs, dict) else obs.fleets
    planets = [Planet(*p) for p in raw_planets]
    fleets = [Fleet(*f) for f in raw_fleets]
    angular_velocity = obs.get("angular_velocity", 0)
    comet_ids = set(obs.get("comet_planet_ids", []) if isinstance(obs, dict) else obs.comet_planet_ids)

    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets
               if p.owner != player
               and p.id not in comet_ids]

    if not targets:
        targets = [p for p in planets if p.owner != player]

    cx, cy = 50, 50

    # 自分の艦隊が既に向かっている惑星のIDを把握
    en_route_targets = {}
    for f in fleets:
        if f.owner == player:
            en_route_targets[f.from_planet_id] = en_route_targets.get(f.from_planet_id, 0) + f.ships

    for mine in my_planets:
        def priority(t):
            dist = math.hypot(mine.x-t.x, mine.y-t.y)
            ships_needed = max(t.ships + 1, 8)
            fleet_speed = 1.0 + 5.0 * (math.log(max(1, ships_needed)) / math.log(1000)) ** 1.5
            eta = dist / fleet_speed
            enemy_produced = eta * t.production if t.owner != -1 else 0
            enemy_bonus = t.production if t.owner != -1 else 0

            # 既に十分な艦隊が向かっている惑星は優先度を下げる
            ships_en_route = en_route_targets.get(t.id, 0)
            needed_now = t.ships + 1
            if ships_en_route >= needed_now:
                return 999999  # 最低優先度

            score = (
                (100 - dist)
                + (30 * t.production)
                + (20 * enemy_bonus)
                - (0.7 * (t.ships + enemy_produced))
                - (2 * eta)
            )
            return -score

        nearest = min(targets, key=priority)

        base_ships = nearest.ships + 1
        fleet_speed = 1.0 + 5.0 * (math.log(max(1, base_ships)) / math.log(1000)) ** 1.5
        dist = math.hypot(mine.x-nearest.x, mine.y-nearest.y)
        eta = dist / fleet_speed
        extra = int(eta * nearest.production) if nearest.owner != -1 else 0
        ships_needed = max(base_ships + extra, 8)

        dist_from_sun = math.hypot(nearest.x-cx, nearest.y-cy)
        is_orbiting = dist_from_sun + nearest.radius < 50

        if is_orbiting:
            turns = int(dist / 3) + 1
            dx = nearest.x - cx
            dy = nearest.y - cy
            r = math.sqrt(dx**2 + dy**2)
            cur_angle = math.atan2(dy, dx)
            fut_angle = cur_angle + angular_velocity * turns
            target_x = cx + r * math.cos(fut_angle)
            target_y = cy + r * math.sin(fut_angle)
        else:
            target_x, target_y = nearest.x, nearest.y

        GUARD = 10
        if mine.ships >= ships_needed + GUARD:
            angle = math.atan2(target_y - mine.y, target_x - mine.x)
            moves.append([mine.id, angle, ships_needed])

    return moves

In [11]:
# グローバル変数（ゲーム開始時にリセット）
_fleet_log = []  # {target_id, ships, arrive_tick}
_step = [0]

def agent_v9(obs, config=None):
    global _fleet_log, _step
    import math
    moves = []

    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]
    angular_velocity = obs.get("angular_velocity", 0)
    comet_ids = set(obs.get("comet_planet_ids", []) if isinstance(obs, dict) else obs.comet_planet_ids)
    current_step = obs.get("step", 0) if isinstance(obs, dict) else getattr(obs, "step", 0)

    # ゲーム開始時にリセット
    if current_step == 0:
        _fleet_log.clear()
        _step[0] = 0

    # 期限切れの艦隊ログを削除
    _fleet_log = [f for f in _fleet_log if f["arrive_tick"] > current_step]

    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets
               if p.owner != player
               and p.id not in comet_ids]

    if not targets:
        targets = [p for p in planets if p.owner != player]

    cx, cy = 50, 50

    # 既に向かっている艦隊の船数を集計
    en_route = {}  # {target_id: ships}
    for f in _fleet_log:
        en_route[f["target_id"]] = en_route.get(f["target_id"], 0) + f["ships"]

    for mine in my_planets:
        def priority(t):
            dist = math.hypot(mine.x-t.x, mine.y-t.y)
            ships_needed = max(t.ships + 1, 8)
            fleet_speed = 1.0 + 5.0 * (math.log(max(1, ships_needed)) / math.log(1000)) ** 1.5
            eta = dist / fleet_speed
            enemy_produced = eta * t.production if t.owner != -1 else 0
            enemy_bonus = t.production if t.owner != -1 else 0

            # 既に十分な艦隊が向かっていればスコアを下げる
            already = en_route.get(t.id, 0)
            needed = t.ships + 1
            if already >= needed:
                return 999999

            score = (
                (100 - dist)
                + (30 * t.production)
                + (20 * enemy_bonus)
                - (0.7 * (t.ships + enemy_produced))
                - (2 * eta)
            )
            return -score

        nearest = min(targets, key=priority)
        if priority(nearest) == 999999:
            continue

        base_ships = nearest.ships + 1
        fleet_speed = 1.0 + 5.0 * (math.log(max(1, base_ships)) / math.log(1000)) ** 1.5
        dist = math.hypot(mine.x-nearest.x, mine.y-nearest.y)
        eta = dist / fleet_speed
        extra = int(eta * nearest.production) if nearest.owner != -1 else 0

        # 既に向かっている分を引いた必要船数
        already = en_route.get(nearest.id, 0)
        ships_needed = max(base_ships + extra - already, 8)

        dist_from_sun = math.hypot(nearest.x-cx, nearest.y-cy)
        is_orbiting = dist_from_sun + nearest.radius < 50

        if is_orbiting:
            turns = int(dist / 3) + 1
            dx = nearest.x - cx
            dy = nearest.y - cy
            r = math.sqrt(dx**2 + dy**2)
            cur_angle = math.atan2(dy, dx)
            fut_angle = cur_angle + angular_velocity * turns
            target_x = cx + r * math.cos(fut_angle)
            target_y = cy + r * math.sin(fut_angle)
        else:
            target_x, target_y = nearest.x, nearest.y

        GUARD = 10
        if mine.ships >= ships_needed + GUARD:
            angle = math.atan2(target_y - mine.y, target_x - mine.x)
            moves.append([mine.id, angle, ships_needed])

            # 艦隊ログに記録
            arrive_tick = current_step + int(dist / fleet_speed) + 1
            _fleet_log.append({
                "target_id": nearest.id,
                "ships": ships_needed,
                "arrive_tick": arrive_tick
            })
            en_route[nearest.id] = en_route.get(nearest.id, 0) + ships_needed

    return moves

In [12]:
# グローバル変数（ゲーム開始時にリセット）
_fleet_log = []  # {target_id, ships, arrive_tick}
_step = [0]

def agent_v10(obs, config=None):
    global _fleet_log, _step
    import math

    moves = []

    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]

    angular_velocity = obs.get("angular_velocity", 0) \
        if isinstance(obs, dict) else obs.angular_velocity

    comet_ids = set(
        obs.get("comet_planet_ids", [])
        if isinstance(obs, dict)
        else obs.comet_planet_ids
    )

    current_step = obs.get("step", 0) \
        if isinstance(obs, dict) else getattr(obs, "step", 0)

    # ゲーム開始時にリセット
    if current_step == 0:
        _fleet_log.clear()
        _step[0] = 0

    # 期限切れ艦隊を削除
    _fleet_log = [
        f for f in _fleet_log
        if f["arrive_tick"] > current_step
    ]

    my_planets = [p for p in planets if p.owner == player]

    targets = [
        p for p in planets
        if p.owner != player
        and p.id not in comet_ids
    ]

    # 全部 comet の場合
    if not targets:
        targets = [p for p in planets if p.owner != player]

    # 太陽中心
    cx, cy = 50, 50

    # 既に向かっている艦隊数
    en_route = {}

    for f in _fleet_log:
        en_route[f["target_id"]] = (
            en_route.get(f["target_id"], 0)
            + f["ships"]
        )

    for mine in my_planets:

        def priority(t):

            dist = math.hypot(mine.x - t.x, mine.y - t.y)

            ships_needed = max(t.ships + 1, 8)

            fleet_speed = (
                1.0
                + 5.0
                * (
                    math.log(max(1, ships_needed))
                    / math.log(1000)
                ) ** 1.5
            )

            eta = dist / fleet_speed

            enemy_produced = (
                eta * t.production
                if t.owner != -1
                else 0
            )

            enemy_bonus = (
                t.production
                if t.owner != -1
                else 0
            )

            # 既に十分向かっているなら除外
            already = en_route.get(t.id, 0)
            needed = t.ships + 1

            if already >= needed:
                return 999999

            score = (
                (100 - dist)
                + (30 * t.production)
                + (20 * enemy_bonus)
                - (0.7 * (t.ships + enemy_produced))
                - (2 * eta)
            )

            return -score

        nearest = min(targets, key=priority)

        if priority(nearest) == 999999:
            continue

        base_ships = nearest.ships + 1

        dist = math.hypot(
            mine.x - nearest.x,
            mine.y - nearest.y
        )

        # 必要船数で速度計算
        fleet_speed = (
            1.0
            + 5.0
            * (
                math.log(max(1, base_ships))
                / math.log(1000)
            ) ** 1.5
        )

        eta = dist / fleet_speed

        extra = (
            int(eta * nearest.production)
            if nearest.owner != -1
            else 0
        )

        already = en_route.get(nearest.id, 0)

        ships_needed = max(
            base_ships + extra - already,
            8
        )

        # 実際に送る船数で再計算
        fleet_speed = (
            1.0
            + 5.0
            * (
                math.log(max(1, ships_needed))
                / math.log(1000)
            ) ** 1.5
        )

        eta = dist / fleet_speed

        # 惑星が軌道上か判定
        dist_from_sun = math.hypot(
            nearest.x - cx,
            nearest.y - cy
        )

        is_orbiting = (
            dist_from_sun + nearest.radius < 50
        )

        if is_orbiting:

            dx = nearest.x - cx
            dy = nearest.y - cy

            r = math.sqrt(dx**2 + dy**2)

            cur_angle = math.atan2(dy, dx)

            # ← 超重要修正
            # 実際のETAを使用
            turns = max(1, round(eta))

            # ← 微調整（当たりやすくなることが多い）
            lead = angular_velocity * (turns - 0.5)

            future_angle = cur_angle + lead

            # ← 半径を考慮して少し内側を狙う
            aim_r = max(0, r - nearest.radius * 0.25)

            target_x = (
                cx
                + aim_r * math.cos(future_angle)
            )

            target_y = (
                cy
                + aim_r * math.sin(future_angle)
            )

        else:
            target_x, target_y = nearest.x, nearest.y

        GUARD = 10

        if mine.ships >= ships_needed + GUARD:

            angle = math.atan2(
                target_y - mine.y,
                target_x - mine.x
            )

            moves.append([
                mine.id,
                angle,
                ships_needed
            ])

            # 艦隊ログ記録
            arrive_tick = (
                current_step
                + max(1, round(eta))
            )

            _fleet_log.append({
                "target_id": nearest.id,
                "ships": ships_needed,
                "arrive_tick": arrive_tick
            })

            en_route[nearest.id] = (
                en_route.get(nearest.id, 0)
                + ships_needed
            )

    return moves

## 4. 実装後の評価

### バージョンアップ前後のエージェントの対戦結果

In [13]:
v2_wins = 0
for i in range(10):
    env = make("orbit_wars", configuration={"seed": i}, debug=False)
    env.run([agent_v2, agent_v1])
    result = env.steps[-1]
    if result[0].reward > result[1].reward:
        v2_wins += 1
print(f"v2 vs v1: {v2_wins}/10")

v2 vs v1: 7/10


In [14]:
v3_wins = 0
for i in range(10):
    env = make("orbit_wars", configuration={"seed": i}, debug=False)
    env.run([agent_v3, agent_v2])
    result = env.steps[-1]
    if result[0].reward > result[1].reward:
        v3_wins += 1
print(f"v3 vs v2: {v3_wins}/10")

v3 vs v2: 10/10


In [15]:
v4_wins = 0
for i in range(10):
    env = make("orbit_wars", configuration={"seed": i}, debug=False)
    env.run([agent_v4, agent_v3])
    result = env.steps[-1]
    if result[0].reward > result[1].reward:
        v4_wins += 1
print(f"v4 vs v3: {v4_wins}/10")

v4 vs v3: 1/10


In [16]:
for name, agent_test in [("A2(生産力50)", agent_v5a2),
                          ("A3(敵ボーナス40)", agent_v5a3),
                          ("A4(ships影響0.3)", agent_v5a4)]:
    wins_v5a = 0
    for i in range(10):
        env = make("orbit_wars", configuration={"seed": i}, debug=False)
        env.run([agent_test, agent_v5a])
        if env.steps[-1][0].reward > env.steps[-1][1].reward:
            wins_v5a += 1
    print(f"{name} vs v5a: {wins_v5a}/10")

A2(生産力50) vs v5a: 1/10
A3(敵ボーナス40) vs v5a: 3/10
A4(ships影響0.3) vs v5a: 5/10


In [17]:
for name, agent_test in [("v5a(生産力重視)", agent_v5a),
                          ("v5b(距離重視)", agent_v5b),
                          ("v5c(到着時間重視)", agent_v5c)]:
    wins_r = 0
    wins_v4 = 0
    for i in range(10):
        env = make("orbit_wars", configuration={"seed": i}, debug=False)
        env.run([agent_test, "random"])
        if env.steps[-1][0].reward > env.steps[-1][1].reward:
            wins_r += 1
    for i in range(10):
        env = make("orbit_wars", configuration={"seed": i}, debug=False)
        env.run([agent_test, agent_v4])
        if env.steps[-1][0].reward > env.steps[-1][1].reward:
            wins_v4 += 1
    print(f"{name}: vs random {wins_r}/10, vs v4 {wins_v4}/10")

v5a(生産力重視): vs random 10/10, vs v4 9/10
v5b(距離重視): vs random 10/10, vs v4 8/10
v5c(到着時間重視): vs random 10/10, vs v4 7/10


In [18]:
v6_wins_r = 0
v6_wins_v5a = 0
for i in range(10):
    env = make("orbit_wars", configuration={"seed": i}, debug=False)
    env.run([agent_v6, "random"])
    if env.steps[-1][0].reward > env.steps[-1][1].reward:
        v6_wins_r += 1

for i in range(10):
    env = make("orbit_wars", configuration={"seed": i}, debug=False)
    env.run([agent_v6, agent_v5a])
    if env.steps[-1][0].reward > env.steps[-1][1].reward:
        v6_wins_v5a += 1

print(f"v6 vs random: {v6_wins_r}/10")
print(f"v6 vs v5a:    {v6_wins_v5a}/10")

v6 vs random: 10/10
v6 vs v5a:    7/10


In [19]:
v7_wins_r = 0
v7_wins_v6 = 0
for i in range(10):
    env = make("orbit_wars", configuration={"seed": i}, debug=False)
    env.run([agent_v7, "random"])
    if env.steps[-1][0].reward > env.steps[-1][1].reward:
        v7_wins_r += 1

for i in range(10):
    env = make("orbit_wars", configuration={"seed": i}, debug=False)
    env.run([agent_v7, agent_v6])
    if env.steps[-1][0].reward > env.steps[-1][1].reward:
        v7_wins_v6 += 1

print(f"v7 vs random: {v7_wins_r}/10")
print(f"v7 vs v6:     {v7_wins_v6}/10")

v7 vs random: 10/10
v7 vs v6:     7/10


In [20]:
v8_wins_r = 0
v8_wins_v7 = 0
for i in range(10):
    env = make("orbit_wars", configuration={"seed": i}, debug=False)
    env.run([agent_v8, "random"])
    if env.steps[-1][0].reward > env.steps[-1][1].reward:
        v8_wins_r += 1

for i in range(10):
    env = make("orbit_wars", configuration={"seed": i}, debug=False)
    env.run([agent_v8, agent_v7])
    if env.steps[-1][0].reward > env.steps[-1][1].reward:
        v8_wins_v7 += 1

print(f"v8 vs random: {v8_wins_r}/10")
print(f"v8 vs v7:     {v8_wins_v7}/10")

v8 vs random: 10/10
v8 vs v7:     2/10


In [21]:
v9_wins_r = 0
v9_wins_v7 = 0
for i in range(10):
    env = make("orbit_wars", configuration={"seed": i}, debug=False)
    env.run([agent_v9, "random"])
    if env.steps[-1][0].reward > env.steps[-1][1].reward:
        v9_wins_r += 1

for i in range(10):
    env = make("orbit_wars", configuration={"seed": i}, debug=False)
    env.run([agent_v9, agent_v7])
    if env.steps[-1][0].reward > env.steps[-1][1].reward:
        v9_wins_v7 += 1

print(f"v9 vs random: {v9_wins_r}/10")
print(f"v9 vs v7:     {v9_wins_v7}/10")

v9 vs random: 10/10
v9 vs v7:     7/10


In [22]:
v10_wins_r = 0
v10_wins_v9 = 0
for i in range(10):
    env = make("orbit_wars", configuration={"seed": i}, debug=False)
    env.run([agent_v10, "random"])
    if env.steps[-1][0].reward > env.steps[-1][1].reward:
        v10_wins_r += 1

for i in range(10):
    env = make("orbit_wars", configuration={"seed": i}, debug=False)
    env.run([agent_v10, agent_v9])
    if env.steps[-1][0].reward > env.steps[-1][1].reward:
        v10_wins_v9 += 1

print(f"v10 vs random: {v10_wins_r}/10")
print(f"v10 vs v9:     {v10_wins_v9}/10")

v10 vs random: 10/10
v10 vs v9:     8/10


### 分析
勝率比較からエージェントの性能がバージョンアップによって改善されていることがわかる。

#### v4時点での問題点
* 自転する惑星をほとんど取れずに負ける→内側は生産力が高いため
* 太陽にあたって自滅する場合がある→太陽回避の動きを組み込むと逆に性能が悪化した
* 自転惑星もたまたまあたって取れることがある→この場合は勝ちやすい
* 外側惑星の生産力が低いマップは負けやすい→生産力の差で負ける

#### v7時点での問題点
* 内側を取れても初速が速い相手に負ける
* 惑星を狙いきれていない（たまにはずす）
* 同じ惑星に戦艦を送りすぎてしまう

#### v10時点での問題点
* 生産性を求めすぎて遠くの惑星から攻めてしまう
* 制圧に必要な戦艦の数が多い盤面では初速が遅くなってしまう

## 5. submission.pyの出力

In [24]:
%%writefile submission.py
import math
from kaggle_environments.envs.orbit_wars.orbit_wars import Planet, Fleet

_fleet_log = []
_step = [0]

def agent(obs, config=None):
    global _fleet_log, _step

    moves = []
    player = obs.get("player", 0) if isinstance(obs, dict) else obs.player
    raw_planets = obs.get("planets", []) if isinstance(obs, dict) else obs.planets
    planets = [Planet(*p) for p in raw_planets]
    angular_velocity = obs.get("angular_velocity", 0) if isinstance(obs, dict) else obs.angular_velocity
    comet_ids = set(obs.get("comet_planet_ids", []) if isinstance(obs, dict) else obs.comet_planet_ids)
    current_step = obs.get("step", 0) if isinstance(obs, dict) else getattr(obs, "step", 0)

    if current_step == 0:
        _fleet_log.clear()
        _step[0] = 0

    _fleet_log = [f for f in _fleet_log if f["arrive_tick"] > current_step]

    my_planets = [p for p in planets if p.owner == player]
    targets = [p for p in planets if p.owner != player and p.id not in comet_ids]

    if not targets:
        targets = [p for p in planets if p.owner != player]

    cx, cy = 50, 50

    en_route = {}
    for f in _fleet_log:
        en_route[f["target_id"]] = en_route.get(f["target_id"], 0) + f["ships"]

    for mine in my_planets:
        def priority(t):
            dist = math.hypot(mine.x-t.x, mine.y-t.y)
            ships_needed = max(t.ships + 1, 8)
            fleet_speed = 1.0 + 5.0 * (math.log(max(1, ships_needed)) / math.log(1000)) ** 1.5
            eta = dist / fleet_speed
            enemy_produced = eta * t.production if t.owner != -1 else 0
            enemy_bonus = t.production if t.owner != -1 else 0
            already = en_route.get(t.id, 0)
            if already >= t.ships + 1:
                return 999999
            score = (
                (100 - dist)
                + (30 * t.production)
                + (20 * enemy_bonus)
                - (0.7 * (t.ships + enemy_produced))
                - (2 * eta)
            )
            return -score

        nearest = min(targets, key=priority)
        if priority(nearest) == 999999:
            continue

        base_ships = nearest.ships + 1
        dist = math.hypot(mine.x-nearest.x, mine.y-nearest.y)
        fleet_speed = 1.0 + 5.0 * (math.log(max(1, base_ships)) / math.log(1000)) ** 1.5
        eta = dist / fleet_speed
        extra = int(eta * nearest.production) if nearest.owner != -1 else 0
        already = en_route.get(nearest.id, 0)
        ships_needed = max(base_ships + extra - already, 8)

        fleet_speed = 1.0 + 5.0 * (math.log(max(1, ships_needed)) / math.log(1000)) ** 1.5
        eta = dist / fleet_speed

        dist_from_sun = math.hypot(nearest.x-cx, nearest.y-cy)
        is_orbiting = dist_from_sun + nearest.radius < 50

        if is_orbiting:
            dx = nearest.x - cx
            dy = nearest.y - cy
            r = math.sqrt(dx**2 + dy**2)
            cur_angle = math.atan2(dy, dx)
            turns = max(1, round(eta))
            lead = angular_velocity * (turns - 0.5)
            future_angle = cur_angle + lead
            aim_r = max(0, r - nearest.radius * 0.25)
            target_x = cx + aim_r * math.cos(future_angle)
            target_y = cy + aim_r * math.sin(future_angle)
        else:
            target_x, target_y = nearest.x, nearest.y

        GUARD = 10
        if mine.ships >= ships_needed + GUARD:
            angle = math.atan2(target_y - mine.y, target_x - mine.x)
            moves.append([mine.id, angle, ships_needed])
            arrive_tick = current_step + max(1, round(eta))
            _fleet_log.append({
                "target_id": nearest.id,
                "ships": ships_needed,
                "arrive_tick": arrive_tick
            })
            en_route[nearest.id] = en_route.get(nearest.id, 0) + ships_needed

    return moves

Overwriting submission.py
